# 03 - Keras Baseline + hls4ml

Este notebook treina um MLP simples sobre a matriz de features gerada no notebook 01/02
e converte o modelo para C++ via hls4ml para posterior síntese na FPGA.

**Seções 1-4**: podem ser executadas no macOS.

**Seções 5-6**: requerem servidor Linux com Vitis HLS 2022.1 instalado e no PATH.

## 1. Imports e Dados

In [ ]:
import os
import platform
import subprocess
from pathlib import Path
import sys

import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Activation, Input
from tensorflow.keras.optimizers import Adam

import hls4ml

PROJECT_ROOT = Path.cwd().parent
SRC_ROOT = PROJECT_ROOT / "src"
DATA_ROOT = PROJECT_ROOT / "data"
sys.path.insert(0, str(SRC_ROOT))

from emg_hls4ml_mvp.dataset import build_feature_dataset

## 2. Treinar MLP Baseline

In [ ]:
subject      = "Sub001"
recording_id = "Sub001_1_05_450_0"
target_column = "index_z"

X, y, _ = build_feature_dataset(DATA_ROOT, subject, recording_id, target_column=target_column)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

model = Sequential([
    Input(shape=(X_train_scaled.shape[1],)),
    Dense(32, name="fc1"),
    Activation("relu", name="relu1"),
    Dense(1, name="output"),
    Activation("linear", name="linear_out"),
])

model.compile(optimizer=Adam(learning_rate=0.001), loss="mse")
model.summary()

history = model.fit(
    X_train_scaled, y_train,
    epochs=30,
    batch_size=32,
    validation_split=0.2,
    verbose=1,
)

In [ ]:
keras_pred = model.predict(X_test_scaled).flatten()

mse  = mean_squared_error(y_test, keras_pred)
rmse = np.sqrt(mse)
mae  = mean_absolute_error(y_test, keras_pred)
r2   = r2_score(y_test, keras_pred)

print(f"MSE:  {mse:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"MAE:  {mae:.4f}")
print(f"R2:   {r2:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].plot(history.history["loss"], label="treino")
axes[0].plot(history.history["val_loss"], label="validação")
axes[0].set_xlabel("Época")
axes[0].set_ylabel("MSE")
axes[0].set_title("Curva de Loss")
axes[0].legend()

n = min(200, len(y_test))
axes[1].plot(y_test[:n], label="Real", linewidth=2)
axes[1].plot(keras_pred[:n], label="Keras (MLP)", linestyle="dashed")
axes[1].set_xlabel("Tempo (frames a 100Hz)")
axes[1].set_ylabel("Ângulo")
axes[1].set_title("Predição Keras vs Real")
axes[1].legend()

plt.tight_layout()
plt.show()

## 3. Configuração do PATH do Vitis HLS ⚠️ Linux

> **A partir daqui: executar no servidor Linux.**
> Garanta que o source do settings64.sh já foi feito **no terminal** antes de iniciar o Jupyter:
> ```bash
> source /home/jeison/Xilinx/Vivado/2022.1/settings64.sh
> source .venv/bin/activate
> jupyter notebook
> ```
>
> A célula abaixo garante que o PATH está correto mesmo que o Jupyter tenha sido iniciado sem o source.

In [ ]:
system = platform.system()

if system == "Linux":
    # Carregar variáveis de ambiente do Vitis HLS 2022.1
    cmd = """
    source /home/jeison/Xilinx/Vivado/2022.1/settings64.sh
    env
    """
    proc = subprocess.Popen(
        ["bash", "-c", cmd],
        stdout=subprocess.PIPE,
        text=True
    )
    for line in proc.stdout:
        key, _, value = line.partition("=")
        os.environ[key.strip()] = value.strip()

    # hls4ml backend='Vivado' procura por 'vivado_hls'.
    # No Vivado 2022.1 o executável se chama 'vitis_hls'.
    # Um wrapper fixo em ~/.local/bin/vivado_hls (script, não symlink)
    # já redireciona para 'vitis_hls' preservando o $0 correto,
    # necessário porque o script vitis_hls resolve caminhos internos
    # via dirname($0) e quebra se chamado via symlink.
    local_bin = Path.home() / ".local" / "bin"
    local_bin_str = str(local_bin)

    # Garantir que ~/.local/bin está no PATH
    if local_bin_str not in os.environ.get("PATH", ""):
        os.environ["PATH"] = local_bin_str + ":" + os.environ["PATH"]

print("Sistema:", system)
print("vivado_hls encontrado:", os.system("command -v vivado_hls > /dev/null") == 0)

## 4. Geração do Projeto C++ via hls4ml

Converte o modelo Keras para um projeto C++ de HLS usando Vitis HLS como backend.

Parâmetros relevantes:
- `Precision: fixed<16,6>` — 16 bits totais, 6 para a parte inteira.
- `ReuseFactor: 1` — máximo paralelismo, máximo recurso.
- `Part` — FPGA alvo: `xc7z020clg400-1` (Zybo Z7-20 / XC7Z020-1CLG400C).

In [ ]:
hls_config = hls4ml.utils.config_from_keras_model(model, granularity="name")

# Configurar todas as camadas
for layer in hls_config["LayerName"].keys():
    hls_config["LayerName"][layer]["Strategy"] = "Latency"
    hls_config["LayerName"][layer]["ReuseFactor"] = 1

hls_config["Model"]["Precision"] = "fixed<16,6>"

print("Configuração hls4ml:")
print(hls_config)

# Backend 'Vivado' é o correto para Vivado/Vitis HLS 2022.1
# (usa o executável 'vivado_hls', que foi mapeado via symlink para 'vitis_hls')
hls_model = hls4ml.converters.convert_from_keras_model(
    model,
    hls_config=hls_config,
    output_dir="model_1/hls4ml_prj",
    backend="Vivado",
    part="xc7z020clg400-1",  # Zybo Z7-20 (XC7Z020-1CLG400C)
    clock_period=10,          # 100 MHz
)
hls_model.write()

print("Projeto C++ gerado em model_1/hls4ml_prj/")

## 5. C-Simulation e Validação Numérica ⚠️ Linux

> O `hls_model.compile()` usa GCC para emular a aritmética de ponto fixo do FPGA.
> Não funciona no macOS (conflito Apple Clang vs headers ap_types da Xilinx).

O objetivo é medir o **erro de quantização**: diferença entre a predição em ponto flutuante
(Keras, float32) e a predição em ponto fixo (hls4ml, fixed<16,6>).

In [ ]:
# Execute no servidor Linux.
hls_model.compile()

hls_pred = hls_model.predict(X_test_scaled).flatten()

quant_mse = mean_squared_error(keras_pred, hls_pred)
print(f"MSE de quantização (Keras float32 vs HLS fixed<16,6>): {quant_mse:.6f}")

n = min(200, len(y_test))
plt.figure(figsize=(12, 4))
plt.plot(y_test[:n],     label="Real",              linewidth=2)
plt.plot(keras_pred[:n], label="Keras (float32)",   linestyle="dashed")
plt.plot(hls_pred[:n],   label="HLS (fixed<16,6>)", linestyle="dotted")
plt.xlabel("Tempo (frames a 100Hz)")
plt.ylabel("Ângulo")
plt.title("Real vs Keras vs HLS")
plt.legend()
plt.tight_layout()
plt.show()

## 6. Síntese de Hardware ⚠️ Linux + Vitis HLS 2022.1

> `hls_model.build()` invoca o Vitis HLS para converter o C++ em RTL (Verilog/VHDL)
> e gerar os relatórios de síntese.

O relatório conterá:
- **Latência estimada** (clock cycles)
- **DSP48E** utilizados
- **LUT** e **FF** utilizados
- **BRAM** utilizados

Com esses números é possível comparar contra os recursos disponíveis na Zybo Z7-20.

In [ ]:
# Execute no servidor Linux com Vivado/Vitis HLS 2022.1 no PATH.
# csim=False pula a C-Simulation (já feita na seção 5).
# export=False não exporta IP (só síntese).
hls_model.build(csim=False, export=False)
print("Síntese concluída.")

## 7. Leitura do Relatório de Síntese

In [ ]:
# Execute no servidor Linux após o build.
report = hls4ml.report.read_vivado_report("model_1/hls4ml_prj")
hls4ml.report.print_vivado_report(report)